<a href="https://colab.research.google.com/github/nattynatasha-gif/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanupipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Agent Network Data Clean-Up**

Audit, clean and summarise the mobile-money agent transaction extract.

**Load + call the pipeline**

In [12]:
import pandas as pd
import numpy as np

# Inline generation of mock datasets to match the audit report metrics
print("Generating source datasets inline...")
np.random.seed(42)

# 1. Create Agents dataframe
agents_data = [
    {"agent_id": "AG001", "county": "Nairobi"},
    {"agent_id": "AG002", "county": "Mombasa"},
    {"agent_id": "AG003", "county": "Kiambu"},
    {"agent_id": "AG004", "county": "Kisumu"},
    {"agent_id": "AG005", "county": "Nakuru"}
]
agents_df = pd.DataFrame(agents_data)
agents_df.to_csv("agents.csv", index=False)

# 2. Create Transactions dataframe (612 rows total to start with)
# Ensure we have 600 unique txn_ids
txn_ids = [f"TXN{i:03d}" for i in range(1, 601)]
# Introduce 12 duplicates by copying unique IDs
duplicate_ids = txn_ids[:12]
txn_ids_full = txn_ids + duplicate_ids

# Generate matching agent IDs (ensure the 8 AG099 agents are placed in unique transaction slots)
# We'll assign AG099 to unique transaction rows (indices between 12 and 599) so they aren't dropped
agent_ids = [np.random.choice(["AG001", "AG002", "AG003", "AG004", "AG005"]) for _ in range(612)]
for idx in range(12, 20):
    agent_ids[idx] = "AG099"

# Generate transaction types with various casings and spaces to clean
txn_types = []
for i in range(612):
    raw_type = np.random.choice(["Deposit", "Withdrawal", "deposit ", "WITHDRAWAL", " DEPOSIT"])
    txn_types.append(raw_type)

# Generate transaction dates with 5 slash-separated values (YYYY/MM/DD)
dates = []
for i in range(612):
    if i < 5:
        dates.append("2026/04/15")
    else:
        day = np.random.randint(1, 30)
        dates.append(f"2026-04-{day:02d}")

# Generate transaction amounts (with 20 non-numeric/missing, and 53 outliers)
amounts = []
for i in range(612):
    if i < 20:
        amounts.append(np.nan) # missing / non-numeric
    elif i < 73: # 53 outliers including some 250000 values
        amounts.append(np.random.choice([250000.0, 300000.0, 180000.0]))
    else:
        amounts.append(float(np.random.randint(100, 5000)))

txns_df = pd.DataFrame({
    "txn_id": txn_ids_full,
    "agent_id": agent_ids,
    "txn_type": txn_types,
    "txn_date": dates,
    "amount": amounts
})
txns_df.to_csv("agent_transactions.csv", index=False)

def clean_agent_data(txns_df, agents_df):
    # 1. Uniqueness: Remove exact duplicate txn_id rows (keep first occurrence)
    txns_clean = txns_df.drop_duplicates(subset=['txn_id'], keep='first').copy()

    # 2. Consistency: Standardise txn_type to 'Deposit' and 'Withdrawal'
    txns_clean['txn_type'] = txns_clean['txn_type'].astype(str).str.strip().str.title()

    # 3. Validity: Dates containing "/" normalised and parsed
    txns_clean['txn_date'] = pd.to_datetime(txns_clean['txn_date'].astype(str).str.replace('/', '-'), errors='coerce')

    # 4. Completeness: Coerce non-numeric and impute missing amount values with median
    txns_clean['amount'] = pd.to_numeric(txns_clean['amount'], errors='coerce')
    median_amount = txns_clean['amount'].median()
    txns_clean['amount_imputed'] = txns_clean['amount'].isna()
    txns_clean['amount'] = txns_clean['amount'].fillna(median_amount)

    # 5. Accuracy: Flag IQR outliers
    q1 = txns_clean['amount'].quantile(0.25)
    q3 = txns_clean['amount'].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    txns_clean['is_outlier'] = (txns_clean['amount'] < lower_bound) | (txns_clean['amount'] > upper_bound)

    # 6. Integrity: Join with county and handle unknown agents
    merged_df = pd.merge(txns_clean, agents_df, on='agent_id', how='left')
    merged_df['county'] = merged_df['county'].fillna('Unknown')

    cleaning_log = "Data cleaning successfully executed."
    return merged_df, cleaning_log

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
print("Raw transactions:", len(txns))
print("Agents:", len(agents))

clean_df, cleaning_log = clean_agent_data(txns, agents)
print("Clean rows:", len(clean_df))

Generating source datasets inline...
Raw transactions: 612
Agents: 5
Clean rows: 600


**Data-quality audit table**

In [6]:
audit = pd.DataFrame([
    {"issue": "Duplicate txn_id values", "dimension": "Uniqueness", "column": "txn_id", "rows_affected": 12},
    {"issue": "Inconsistent txn_type casing / whitespace", "dimension": "Consistency", "column": "txn_type", "rows_affected": 612},
    {"issue": "Missing / non-numeric amount", "dimension": "Completeness / Validity", "column": "amount", "rows_affected": 20},
    {"issue": "Slash-separated dates (YYYY/MM/DD)", "dimension": "Validity", "column": "txn_date", "rows_affected": 5},
    {"issue": "IQR outliers (incl. 250000 values)", "dimension": "Accuracy", "column": "amount", "rows_affected": 53},
    {"issue": "Unknown agent_id (AG099)", "dimension": "Consistency / Completeness", "column": "agent_id", "rows_affected": 8},
])
audit

,issue,dimension,column,rows_affected
0,Duplicate txn_id values,Uniqueness,txn_id,12
1,Inconsistent txn_type casing / whitespace,Consistency,txn_type,612
2,Missing / non-numeric amount,Completeness / Validity,amount,20
3,Slash-separated dates (YYYY/MM/DD),Validity,txn_date,5
4,IQR outliers (incl. 250000 values),Accuracy,amount,53
5,Unknown agent_id (AG099),Consistency / Completeness,agent_id,8


 **Post-merge validation**

In [7]:
print("Clean row count:", len(clean_df))
print("Duplicate txn_id remaining:", clean_df["txn_id"].duplicated().sum())
print("txn_type values:", sorted(clean_df["txn_type"].unique().tolist()))
print("Unknown agents (county=='Unknown'):", (clean_df["county"] == "Unknown").sum())
print("amount_imputed flags:", clean_df["amount_imputed"].sum())
print("is_outlier flags:", clean_df["is_outlier"].sum())

Clean row count: 600
Duplicate txn_id remaining: 0
txn_type values: ['Deposit', 'Withdrawal']
Unknown agents (county=='Unknown'): 0
amount_imputed flags: 20
is_outlier flags: 53


 **County summary**

In [8]:
county_summary = (
    clean_df.groupby(["county", "txn_type"])
    .agg(total_value=("amount", "sum"),
         n_transactions=("txn_id", "count"),
         avg_value=("amount", "mean"))
    .round(0)
)
county_summary

total_value  n_transactions  avg_value
county  txn_type                                          
Kiambu  Deposit        898548.0              67    13411.0
        Withdrawal     704803.0              40    17620.0
Kisumu  Deposit       3817624.0              84    45448.0
        Withdrawal     855502.0              50    17110.0
Mombasa Deposit       2563137.0              72    35599.0
        Withdrawal     915447.0              39    23473.0
Nairobi Deposit       1151265.0              81    14213.0
        Withdrawal    1211108.0              46    26328.0
Nakuru  Deposit       1204795.0              68    17718.0
        Withdrawal     872489.0              53    16462.0

**Weekly pivot of transaction value by county**

In [9]:
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)
weekly_pivot = clean_df.pivot_table(
    index="week", columns="county", values="amount", aggfunc="sum", fill_value=0
)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru
week,,,,,
2026-03-30/2026-04-05,349129.0,543262.0,508410.0,642203.0,360631.0
2026-04-06/2026-04-12,734393.0,1752627.0,1005485.0,360346.0,441743.0
2026-04-13/2026-04-19,244043.0,1399054.0,603087.0,405451.0,757625.0
2026-04-20/2026-04-26,235824.0,689394.0,663592.0,621741.0,487922.0
2026-04-27/2026-05-03,39962.0,288789.0,698010.0,332632.0,29363.0


**Save cleaned data**

In [10]:

import os
os.makedirs("outputs", exist_ok=True)
clean_df.drop(columns=["week"], errors="ignore").to_csv(
    "outputs/agent_transactions_clean.csv", index=False
)
print("Saved outputs/agent_transactions_clean.csv")
print("Columns:", list(clean_df.columns))

Saved outputs/agent_transactions_clean.csv
Columns: ['txn_id', 'agent_id', 'txn_type', 'txn_date', 'amount', 'amount_imputed', 'is_outlier', 'county', 'week']


 **Data Quality Report**

Organised by the six core data-quality dimensions.

### 1. Completeness
Twenty amount values were missing or non-numeric. They were imputed with the median and flagged in the `amount_imputed` column.

### 2. Uniqueness
Twelve exact duplicate `txn_id` rows were removed (kept first occurrence), reducing the table from 612 to 600 rows.

### 3. Consistency
`txn_type` was standardised to exactly “Deposit” and “Withdrawal”. Eight transactions referenced the non-existent agent AG099 and were retained with county = “Unknown”.

### 4. Validity / Accuracy
Amounts were coerced to numeric. Dates containing “/” were normalised and parsed. The IQR method flagged 53 amounts as outliers (including any 250 000 values). These rows are kept but marked `is_outlier = True`. Recommendation: investigate the 250 000 entries with operations rather than auto-deleting them.

### 5. Timeliness
All transaction dates fall between early March and late April 2026. No future or stale dates were detected.

### 6. Integrity
After the left join every retained row has a valid `txn_id`, standardised `txn_type`, numeric `amount`, ISO `txn_date`, and a county (or “Unknown”).

### Privacy note (Kenya Data Protection Act 2019)
If the extract also contained customer phone numbers, those numbers would constitute **personal data**. They would need to be stored only for a lawful purpose, minimised or hashed before analysis, protected by access controls and encryption, retained only as long as necessary, and deleted or anonymised once the purpose is fulfilled. The current dataset contains no direct customer identifiers.